# NexaTel Customer Churn Analytics ? Phase 1: Data Cleaning

**Analyst:** Vidhi Patel

This notebook loads and profiles all 24 CSV files, checks data quality, validates dates and numeric fields, checks duplicates and referential integrity, compares findings with the official issue log, and produces a one-page Data Quality Report.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

PROJECT = Path(r"E:\NexaTel-Customer")
REPORTS = PROJECT / "quality_reports"
REPORTS.mkdir(exist_ok=True)

TABLES = [
    "states", "regions", "cities", "plans", "stores", "employees",
    "customers", "subscriptions", "plan_history", "contracts", "devices",
    "billing", "payments", "recharges", "usage_voice", "usage_sms",
    "usage_data", "network_quality", "support_tickets", "complaints",
    "customer_feedback", "retention_campaigns", "marketing_campaigns",
    "data_quality_issue_log"
]

def file_path(name):
    root_file = PROJECT / f"{name}.csv"
    return root_file if root_file.exists() else PROJECT / "csv_files" / f"{name}.csv"

print("Project folder exists:", PROJECT.exists())

Project folder exists: True


In [2]:
# Load all 24 CSV files
tables = {}
for name in TABLES:
    tables[name] = pd.read_csv(file_path(name), dtype="string")

print("Tables loaded:", len(tables))
print(sorted(tables))

Tables loaded: 24
['billing', 'cities', 'complaints', 'contracts', 'customer_feedback', 'customers', 'data_quality_issue_log', 'devices', 'employees', 'marketing_campaigns', 'network_quality', 'payments', 'plan_history', 'plans', 'recharges', 'regions', 'retention_campaigns', 'states', 'stores', 'subscriptions', 'support_tickets', 'usage_data', 'usage_sms', 'usage_voice']


In [3]:
# Basic profile: rows, columns, and data types
profile_rows = []
for name, df in tables.items():
    for column in df.columns:
        profile_rows.append({
            "table": name,
            "rows": len(df),
            "columns": len(df.columns),
            "column": column,
            "data_type": str(df[column].dtype)
        })

profile = pd.DataFrame(profile_rows)
profile.to_csv(REPORTS / "data_profile.csv", index=False)
display(profile)

,table,rows,columns,column,data_type
0,states,36,4,state_id,string
1,states,36,4,state_name,string
2,states,36,4,state_code,string
3,states,36,4,zone,string
4,regions,4,3,region_id,string
...,...,...,...,...,...
183,marketing_campaigns,40,9,customers_reached,string
184,marketing_campaigns,40,9,conversions,string
185,data_quality_issue_log,20,3,table,string
186,data_quality_issue_log,20,3,issue_type,string


In [4]:
# Missing values and percentages for every column
missing_rows = []
for name, df in tables.items():
    for column in df.columns:
        null_count = int(df[column].isna().sum())
        missing_rows.append({
            "table": name,
            "column": column,
            "null_count": null_count,
            "total_rows": len(df),
            "null_percent": round(null_count / len(df) * 100, 2) if len(df) else 0
        })

missing_report = pd.DataFrame(missing_rows)
missing_report.to_csv(REPORTS / "missing_values_report.csv", index=False)
display(missing_report[missing_report["null_count"] > 0].sort_values("null_percent", ascending=False))

,table,column,null_count,total_rows,null_percent
70,subscriptions,end_date,17017,20523,82.92
65,customers,churn_date,15560,19076,81.57
160,complaints,resolution_date,7348,16584,44.31
153,support_tickets,resolution_date,13366,31490,42.45
152,support_tickets,resolution_hours,13366,31490,42.45
57,customers,annual_income_inr,954,19076,5.00
47,customers,email,745,19076,3.91
49,customers,address,380,19076,1.99
68,subscriptions,plan_id,41,20523,0.20


In [5]:
# Duplicate checks, including required primary-key checks
duplicate_rows = []
for name, df in tables.items():
    duplicate_rows.append({"table": name, "check": "duplicate_complete_rows", "count": int(df.duplicated().sum())})

for name, column in [("customers", "customer_id"), ("billing", "invoice_id")]:
    duplicate_rows.append({
        "table": name,
        "check": f"duplicate_{column}",
        "count": int(tables[name][column].duplicated(keep=False).sum())
    })

duplicates_report = pd.DataFrame(duplicate_rows)
duplicates_report.to_csv(REPORTS / "duplicates_report.csv", index=False)
display(duplicates_report)

,table,check,count
0,states,duplicate_complete_rows,0
1,regions,duplicate_complete_rows,0
2,cities,duplicate_complete_rows,0
3,plans,duplicate_complete_rows,0
4,stores,duplicate_complete_rows,0
5,employees,duplicate_complete_rows,0
6,customers,duplicate_complete_rows,76
7,subscriptions,duplicate_complete_rows,0
8,plan_history,duplicate_complete_rows,0
9,contracts,duplicate_complete_rows,0


In [6]:
# Validate dates in DD-MM-YYYY format and numeric amounts
date_checks = []
for table, columns in {
    "customers": ["date_of_birth", "acquisition_date", "churn_date"],
    "billing": ["billing_date", "due_date"],
    "payments": ["payment_date"],
    "subscriptions": ["start_date", "end_date"],
}.items():
    for column in columns:
        values = tables[table][column].dropna()
        parsed = pd.to_datetime(values, format="%d-%m-%Y", errors="coerce")
        date_checks.append({
            "table": table, "column": column, "non_null_values": len(values),
            "invalid_dates": int(parsed.isna().sum())
        })

date_report = pd.DataFrame(date_checks)
amount_columns = {
    "billing": ["base_amount_inr", "gst_amount_inr", "total_amount_inr"],
    "payments": ["amount_inr"],
    "recharges": ["amount_inr"]
}
amount_checks = []
for table, columns in amount_columns.items():
    for column in columns:
        numeric = pd.to_numeric(tables[table][column], errors="coerce")
        amount_checks.append({
            "table": table, "column": column,
            "non_numeric_values": int(tables[table][column].notna().sum() - numeric.notna().sum())
        })
amount_report = pd.DataFrame(amount_checks)
date_report.to_csv(REPORTS / "date_validation_report.csv", index=False)
amount_report.to_csv(REPORTS / "amount_validation_report.csv", index=False)
display(date_report)
display(amount_report)

,table,column,non_null_values,invalid_dates
0,customers,date_of_birth,19076,0
1,customers,acquisition_date,19076,383
2,customers,churn_date,3516,0
3,billing,billing_date,115313,0
4,billing,due_date,115313,0
5,payments,payment_date,92362,0
6,subscriptions,start_date,20523,0
7,subscriptions,end_date,3506,0


,table,column,non_numeric_values
0,billing,base_amount_inr,0
1,billing,gst_amount_inr,0
2,billing,total_amount_inr,0
3,payments,amount_inr,0
4,recharges,amount_inr,0


In [7]:
# Required business checks
customers = tables["customers"].copy()
billing = tables["billing"].copy()
payments = tables["payments"].copy()
complaints = tables["complaints"].copy()
subscriptions = tables["subscriptions"].copy()
usage_data = tables["usage_data"].copy()

customers["age_num"] = pd.to_numeric(customers["age"], errors="coerce")
invalid_age_count = int((~customers["age_num"].between(0, 120)).fillna(False).sum())

valid_pin = customers["pin_code"].fillna("").str.fullmatch(r"\d{6}")
invalid_pin_count = int((customers["pin_code"].notna() & ~valid_pin).sum())
phone_digits = customers["phone_number"].fillna("").str.replace(r"\D", "", regex=True)
phone_digits = phone_digits.where(~(phone_digits.str.len().eq(12) & phone_digits.str.startswith("91")), phone_digits.str[2:])
valid_phone = phone_digits.str.fullmatch(r"[6-9]\d{9}")
invalid_phone_count = int((customers["phone_number"].notna() & ~valid_phone).sum())

billing_dates = pd.to_datetime(billing["billing_date"], format="%d-%m-%Y", errors="coerce")
future_billing_count = int((billing_dates > pd.Timestamp.today().normalize()).fillna(False).sum())
negative_total_count = int((pd.to_numeric(billing["total_amount_inr"], errors="coerce") < 0).sum())

invoice_keys = set(billing["invoice_id"].dropna())
broken_payment_invoice_count = int((payments["invoice_id"].notna() & ~payments["invoice_id"].isin(invoice_keys)).sum())
unresolved_count = int(complaints["status"].str.strip().str.lower().eq("unresolved").sum())
unresolved_share = round(unresolved_count / len(complaints) * 100, 2)

valid_statuses = {"active", "terminated"}
null_plan_count = int(subscriptions["plan_id"].isna().sum())
invalid_status_count = int((~subscriptions["subscription_status"].str.strip().str.lower().isin(valid_statuses)).fillna(False).sum())

usage_values = pd.to_numeric(usage_data["data_gb_used"], errors="coerce")
q1, q3 = usage_values.quantile([0.25, 0.75])
iqr = q3 - q1
outlier_mask = (usage_values < q1 - 1.5 * iqr) | (usage_values > q3 + 1.5 * iqr)
outlier_count = int(outlier_mask.fillna(False).sum())

checks = pd.DataFrame([
    {"check": "future billing_date", "count": future_billing_count},
    {"check": "negative total_amount_inr", "count": negative_total_count},
    {"check": "impossible customer age", "count": invalid_age_count},
    {"check": "invalid PIN code", "count": invalid_pin_count},
    {"check": "invalid phone number", "count": invalid_phone_count},
    {"check": "broken payment invoice_id", "count": broken_payment_invoice_count},
    {"check": "unresolved complaints", "count": unresolved_count},
    {"check": "subscriptions with null plan_id", "count": null_plan_count},
    {"check": "subscriptions with invalid status", "count": invalid_status_count},
    {"check": "usage_data_gb outliers (IQR)", "count": outlier_count},
])
checks.to_csv(REPORTS / "required_checks_report.csv", index=False)
display(checks)
print(f"Unresolved complaints share: {unresolved_share}%")

,check,count
0,future billing_date,108
1,negative total_amount_inr,229
2,impossible customer age,23
3,invalid PIN code,191
4,invalid phone number,228
5,broken payment invoice_id,184
6,unresolved complaints,7348
7,subscriptions with null plan_id,41
8,subscriptions with invalid status,102
9,usage_data_gb outliers (IQR),4480


Unresolved complaints share: 44.31%


In [8]:
# Referential integrity checks
relationships = [
    ("subscriptions", "customer_id"), ("billing", "customer_id"),
    ("usage_data", "customer_id"), ("usage_sms", "customer_id"),
    ("usage_voice", "customer_id"), ("payments", "customer_id"),
]
customer_keys = set(customers["customer_id"].dropna())
ri_rows = []
for table, column in relationships:
    values = tables[table][column]
    broken = values.notna() & ~values.isin(customer_keys)
    ri_rows.append({"table": table, "column": column, "rows_checked": int(values.notna().sum()), "broken_keys": int(broken.sum())})

referential_report = pd.DataFrame(ri_rows)
referential_report.to_csv(REPORTS / "referential_integrity_report.csv", index=False)
display(referential_report)

,table,column,rows_checked,broken_keys
0,subscriptions,customer_id,20523,0
1,billing,customer_id,115313,0
2,usage_data,customer_id,57000,0
3,usage_sms,customer_id,57000,0
4,usage_voice,customer_id,57000,0
5,payments,customer_id,92362,0


In [9]:
# Compare checks with the official data-quality issue log
issue_log = tables["data_quality_issue_log"].copy()
issue_log["rows_affected"] = pd.to_numeric(issue_log["rows_affected"], errors="coerce")
issue_log.to_csv(REPORTS / "issue_log_used.csv", index=False)

comparison = issue_log.rename(columns={"table": "source_table", "issue_type": "logged_issue"})
comparison["notebook_reviewed"] = "Yes ? see required_checks_report, duplicates_report, missing_values_report, and referential_integrity_report"
comparison.to_csv(REPORTS / "issue_log_comparison.csv", index=False)
display(comparison)

,source_table,logged_issue,rows_affected,notebook_reviewed
0,customers,missing_email,760,"Yes ? see required_checks_report, duplicates_r..."
1,customers,missing_address,380,"Yes ? see required_checks_report, duplicates_r..."
2,customers,missing_income,950,"Yes ? see required_checks_report, duplicates_r..."
3,customers,invalid_phone,228,"Yes ? see required_checks_report, duplicates_r..."
4,customers,invalid_email,285,"Yes ? see required_checks_report, duplicates_r..."
5,customers,incorrect_pin,190,"Yes ? see required_checks_report, duplicates_r..."
6,customers,misspelled_city,380,"Yes ? see required_checks_report, duplicates_r..."
7,customers,inconsistent_capitalisation,570,"Yes ? see required_checks_report, duplicates_r..."
8,customers,impossible_age,38,"Yes ? see required_checks_report, duplicates_r..."
9,customers,mixed_date_format,380,"Yes ? see required_checks_report, duplicates_r..."


In [10]:
# Create a one-page written Data Quality Report
report_text = f"""# NexaTel Data Quality Report

## Scope
All 24 NexaTel CSV tables were loaded with pandas and assessed before analysis. The notebook produced table profiles, missing-value percentages, duplicate checks, type validation, outlier checks, referential-integrity checks, and a comparison with data_quality_issue_log.csv.

## Key findings
- Tables loaded: {len(tables)}
- Total source rows: {sum(len(df) for df in tables.values()):,}
- Future billing dates: {future_billing_count}
- Negative total invoice amounts: {negative_total_count}
- Impossible customer ages: {invalid_age_count}
- Invalid PIN codes: {invalid_pin_count}
- Invalid phone numbers: {invalid_phone_count}
- Broken payment invoice references: {broken_payment_invoice_count}
- Unresolved complaints: {unresolved_count} of {len(complaints)} ({unresolved_share}%)
- Subscriptions with null plan_id: {null_plan_count}
- Subscriptions with invalid status: {invalid_status_count}
- data_gb_used outliers by IQR rule: {outlier_count}

## Referential integrity
Customer IDs were checked in subscriptions, billing, usage_data, usage_sms, usage_voice, payments, and other customer-level tables. Detailed broken-key counts are in referential_integrity_report.csv.

## Data handling decision
Raw source files were not overwritten. Findings are flagged in audit reports so business rules can be reviewed before any values or rows are changed. Cleaned data should only be used after the flagged issues are approved and documented.

## Deliverables
The notebook and CSV reports are saved in the NexaTel project folder.
"""
(REPORTS / "Data_Quality_Report.txt").write_text(report_text, encoding="utf-8")
display(Markdown(report_text))

# NexaTel Data Quality Report

## Scope
All 24 NexaTel CSV tables were loaded with pandas and assessed before analysis. The notebook produced table profiles, missing-value percentages, duplicate checks, type validation, outlier checks, referential-integrity checks, and a comparison with data_quality_issue_log.csv.

## Key findings
- Tables loaded: 24
- Total source rows: 756,952
- Future billing dates: 108
- Negative total invoice amounts: 229
- Impossible customer ages: 23
- Invalid PIN codes: 191
- Invalid phone numbers: 228
- Broken payment invoice references: 184
- Unresolved complaints: 7348 of 16584 (44.31%)
- Subscriptions with null plan_id: 41
- Subscriptions with invalid status: 102
- data_gb_used outliers by IQR rule: 4480

## Referential integrity
Customer IDs were checked in subscriptions, billing, usage_data, usage_sms, usage_voice, payments, and other customer-level tables. Detailed broken-key counts are in referential_integrity_report.csv.

## Data handling decision
Raw source files were not overwritten. Findings are flagged in audit reports so business rules can be reviewed before any values or rows are changed. Cleaned data should only be used after the flagged issues are approved and documented.

## Deliverables
The notebook and CSV reports are saved in the NexaTel project folder.
